<a href="https://colab.research.google.com/github/melissa-04/spink1-gbm-hypoxia/blob/main/03_transcriptome_gsea.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03 Transcriptome-wide correlation and GSEA

Computes the Spearman correlation of every expressed gene with SPINK1 in the discovery cohort and runs pre-ranked GSEA against the 50 MSigDB hallmark gene sets. Produces Table S4 and Table S5.

In [1]:
%%capture
!pip install -q gseapy

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import pandas as pd
import gseapy as gp
from scipy import stats
from statsmodels.stats.multitest import multipletests

REPO = '/content/drive/MyDrive/SPINK1/repo'
RESULTS = os.path.join(REPO, 'results')
L = pd.read_parquet(os.path.join(REPO, 'processed', 'tcga_expression.parquet'))
print(L.shape, '| gseapy', gp.__version__)

Mounted at /content/drive
(17754, 144) | gseapy 1.3.1


In [3]:
R = L.T.rank()
Rc = R - R.mean()
ss = np.sqrt((Rc ** 2).sum())
rho = Rc.mul(Rc['SPINK1'], axis=0).sum() / (ss * ss['SPINK1'])
rho = rho.drop('SPINK1').replace([np.inf, -np.inf], np.nan).dropna()

n = L.shape[1]
t = rho * np.sqrt((n - 2) / (1 - rho ** 2))
s4 = pd.DataFrame({'gene': rho.index, 'rho': rho.values,
                   'p': 2 * stats.t.sf(np.abs(t), df=n - 2)})
s4['FDR_q'] = multipletests(s4['p'], method='fdr_bh')[1]
s4 = s4.sort_values('rho', ascending=False).reset_index(drop=True)
s4.to_csv(os.path.join(RESULTS, 'Table_S4_SPINK1_genomewide_correlation.csv'), index=False)

sig = s4[s4['FDR_q'] < 0.05]
print('Genes tested:', len(s4), '| FDR < 0.05:', len(sig),
      f'({(sig.rho > 0).sum()} positive, {(sig.rho < 0).sum()} negative)')
print(s4.head(10).round(3).to_string(index=False))
print(s4.tail(10).iloc[::-1].round(3).to_string(index=False))

Genes tested: 17753 | FDR < 0.05: 4158 (2581 positive, 1577 negative)
    gene   rho   p  FDR_q
     IL8 0.664 0.0    0.0
   TREM1 0.617 0.0    0.0
 C5orf46 0.597 0.0    0.0
   CCL20 0.594 0.0    0.0
     PI3 0.585 0.0    0.0
C15orf48 0.584 0.0    0.0
   NDRG1 0.583 0.0    0.0
  INSIG2 0.582 0.0    0.0
  S100A9 0.578 0.0    0.0
  ENTHD1 0.568 0.0    0.0
    gene    rho   p  FDR_q
   ATAD2 -0.484 0.0    0.0
    MCM4 -0.468 0.0    0.0
    LIG1 -0.462 0.0    0.0
 GATAD2B -0.459 0.0    0.0
   DNMT1 -0.449 0.0    0.0
   XRCC3 -0.448 0.0    0.0
    MSH2 -0.440 0.0    0.0
   BRIP1 -0.439 0.0    0.0
TIMELESS -0.434 0.0    0.0
    ILF3 -0.431 0.0    0.0


In [4]:
pre = gp.prerank(rnk=s4[['gene', 'rho']], gene_sets='MSigDB_Hallmark_2020',
                 permutation_num=1000, seed=42, min_size=10, max_size=500,
                 outdir=None, verbose=False)

s5 = pre.res2d.copy()
s5['NES'] = pd.to_numeric(s5['NES'])
s5['FDR q-val'] = pd.to_numeric(s5['FDR q-val'])
s5 = s5.sort_values('NES', ascending=False).reset_index(drop=True)
s5 = s5[['Term', 'ES', 'NES', 'NOM p-val', 'FDR q-val', 'Tag %', 'Lead_genes']]
s5.to_csv(os.path.join(RESULTS, 'Table_S5_GSEA_hallmark.csv'), index=False)

print('Gene sets:', len(s5), '| FDR < 0.05:', int((s5['FDR q-val'] < 0.05).sum()))
print(s5[['Term', 'NES', 'FDR q-val']].round(3).to_string(index=False))

2026-10-22 19:13:35,505 [WARNING] Duplicated values found in preranked stats: 1.68% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


Gene sets: 50 | FDR < 0.05: 40
                             Term    NES  FDR q-val
    TNF-alpha Signaling via NF-kB  3.062      0.000
            Inflammatory Response  2.956      0.000
                          Hypoxia  2.801      0.000
         IL-6/JAK/STAT3 Signaling  2.763      0.000
              Allograft Rejection  2.724      0.000
                      Coagulation  2.535      0.000
Epithelial Mesenchymal Transition  2.531      0.000
                       Complement  2.519      0.000
                KRAS Signaling Up  2.334      0.000
                        Apoptosis  2.214      0.000
  Reactive Oxygen Species Pathway  2.189      0.000
             IL-2/STAT5 Signaling  2.185      0.000
                      p53 Pathway  2.067      0.000
                       Glycolysis  2.057      0.000
                     Angiogenesis  2.054      0.000
        Interferon Gamma Response  2.052      0.000
            Xenobiotic Metabolism  2.042      0.000
                Protein Secretion